In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm

from loka.data import load_breakthrough_states
from loka.models import BreakthroughCNN
from loka.geometrical_methods import herding_order, k_center_order

import os
import pandas as pd

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
emb_prefix = "/home/axelbjarkar/RU/T-404-LOKA-code/data/ordered-data/embeddings"

datasets = [
    (
        "k_center_no_S0", 
        lambda : load_breakthrough_states("/home/axelbjarkar/RU/T-404-LOKA-code/data/ordered-data/k_center_no_S0_25p.txt", two_channel=True)
    ),
    (
        "herding",
        lambda : load_breakthrough_states("/home/axelbjarkar/RU/T-404-LOKA-code/data/ordered-data/herding_25p.txt", two_channel=True)
    ),
    (
        "random",
        lambda : load_breakthrough_states("/home/axelbjarkar/RU/T-404-LOKA-code/data/o.txt", two_channel=True)
    ),
    (
        "k_center_rand_S0_25p",
        lambda : load_breakthrough_states("/home/axelbjarkar/RU/T-404-LOKA-code/data/ordered-data/k_center_rand_S0_25p.txt", two_channel=True)
    ),
    (
        "herding_with_counts_10p",
        lambda : load
    )
    
]

filename = "/home/axelbjarkar/RU/T-404-LOKA-code/data/o.txt"
X_all, Y_all = load_breakthrough_states(filename, two_channel=True)
total_data_amount = X_all.shape[0]

In [ ]:
total_data_amount

In [ ]:
start = 0.01
end = 0.1
steps = 50
batch_size = 512
eval_bs = 1024


data_dir = "/home/axelbjarkar/RU/T-404-LOKA-code/data/ordered-data/results/mse_all"
os.makedirs(data_dir, exist_ok=True)  # create the folder if it doesn't exist yet

X_all_t = torch.FloatTensor(X_all).to(device)
Y_all_t = torch.FloatTensor(Y_all).to(device)

for filename, dataset in datasets:
    csv_file = f'{data_dir}/{filename}.csv'
    if os.path.exists(csv_file):
        print(f'skipping {filename}: {csv_file} already exists')
        continue
    
    splits = np.linspace(start, end, num=steps, endpoint=True)
    mse_values = []
    prc_values = []
    
    epochs = 50
    pbar = tqdm(total=steps * epochs, desc='training')
    
    # full dataset on the GPU once, for evaluation
    X,Y = dataset()
    
    X_t = torch.FloatTensor(X).to(device)
    Y_t = torch.FloatTensor(Y).to(device)
    
    for m, ratio in enumerate(splits):
        n_train = int(total_data_amount * ratio)
        X_train, Y_train = X[:n_train], Y[:n_train]  # first `ratio` of the data, in order
    
        model = BreakthroughCNN().to(device)
    
        # training setup
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.MSELoss()
    
        # convert training data to tensors
        X_train_t = torch.FloatTensor(X_train).to(device)
        Y_train_t = torch.FloatTensor(Y_train).to(device)
    
        # training loop
        for epoch in range(epochs):
            perm = torch.randperm(len(X_train_t), device=device)  # new shuffle each epoch
    
            for i in range(0, len(X_train_t), batch_size):
                idx = perm[i:i + batch_size]
                xb, yb = X_train_t[idx], Y_train_t[idx]
    
                optimizer.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                optimizer.step()
            pbar.update(1)
    
        pbar.set_description(f'calculating total MSE ({filename})')
    
        # batched evaluation
        sq_err, n = 0.0, 0
        with torch.no_grad():
            for i in range(0, len(X_all_t), eval_bs):
                xb, yb = X_all_t[i:i + eval_bs], Y_all_t[i:i + eval_bs]
                sq_err += F.mse_loss(model(xb), yb, reduction='sum').item()  # total squared error in this chunk
                n += yb.numel()  # count of values summed (numel = number of elements, all dims)
        mse = sq_err / n # total / count = mean, same as nn.MSELoss()        
        mse_values.append(mse)  
        prc_values.append(ratio)
    
        # free GPU memory before the next model
        del model, optimizer, X_train_t, Y_train_t
        torch.cuda.empty_cache()

        pbar.set_description(f'training {filename}')
        pbar.set_postfix(model=f'{m+1}/{steps}', ratio=f'{ratio:.2f}',
                         loss=f'{loss.item():.4f}', mse=f'{mse:.6f}')
    
    pbar.close()
    
    # one row per split: training ratio and MSE
    pd.DataFrame({'ratio': prc_values, 'mse': mse_values}).to_csv(csv_file, index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))

ax.plot(np.array(prc_values) * 100, mse_values, 'o-', color='#2196F3', linewidth=2, markersize=8, markerfacecolor='white', markeredgewidth=2)

ax.set_xlabel(r"$\frac{|S|}{|E|}$ (%)", fontsize=20)
ax.set_ylabel("MSE", fontsize=20)
ax.set_title("Model Performance vs Training Subset Size", fontsize=22)

ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
data_dir = "/home/axelbjarkar/RU/T-404-LOKA-code/data/ordered-data/results"
os.makedirs(data_dir, exist_ok=True)  # create the folder if it doesn't exist yet

csv_file = f'{data_dir}/k_center_no_S0_25p_correct_order.csv'

# one row per split: training ratio and MSE
pd.DataFrame({'ratio': prc_values, 'mse': mse_values}).to_csv(csv_file, index=False)